# Study 827 — Cross-Asset Skewness Premium 🎲🌐

**Does the single-name "lottery names under-earn" effect carry up to whole *asset classes*?**

The single-name realized-skewness reversal (Amaya, Christoffersen, Jacobs & Vasquez 2015;
Study 803) says the most right-skewed **stocks** go on to earn *less*. Here we ask its
asset-class analogue: measure each of nine asset-class ETFs' **trailing realized skewness**,
each month go **long the low-skew / short the high-skew** classes, and see whether low-skew
classes out-earn. Real tape 2007-01-03 → 2026-06-30, 9 classes.

*Numbers below are the frozen headline (`docs/results.md`, fingerprint `9ce7d7c0e243`); the
live cells run the fast synthetic control. Survivorship: fixed current-membership class-proxy
ETFs — milder than a single-name universe, named on the Signal axis.*


## 1. The idea in one picture

In the stock cross-section, a right-skewed name has a fat *upside* tail — the occasional big up-day — and lottery-loving investors overpay for it, so it under-earns. Do investors do the same *across* asset classes: bidding up whichever class (a commodity spike, an EM melt-up) recently looked most lottery-like? Sort the nine classes on their trailing third moment; buy the boring low-skew ones, sell the lottery-like high-skew ones.

In [1]:
import numpy as np, pandas as pd
R = dict(spread_bps=13.73, t_nw=0.62, lo_bps=63.24, hi_bps=49.51, sharpe=0.15)
print('long low-skew / short high-skew spread: %+.2f bps/month (NW t = %+.2f)'
      % (R['spread_bps'], R['t_nw']))
print('  low-skew book %+.2f bps vs high-skew book %+.2f bps'
      % (R['lo_bps'], R['hi_bps']))
print('  gross spread Sharpe (before cost, annualised): %.2f' % R['sharpe'])

long low-skew / short high-skew spread: +13.73 bps/month (NW t = +0.62)
  low-skew book +63.24 bps vs high-skew book +49.51 bps
  gross spread Sharpe (before cost, annualised): 0.15


## 2. Is the sort just lucky? A live synthetic control

We plant the effect in a seeded toy world of nine synthetic classes (`edge>0`) and check the detector recovers it — and that it stays *silent* on the null (`edge=0`, skew present but unpriced). No network.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from cross_asset_skew import data, strategy as st
null = st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=827, n_assets=9, n_days=3000))
planted = st.synthetic_detect(data.synthetic_panel(edge=0.004, seed=827, n_assets=9, n_days=4000))
print('null world   : spread NW t = %+.2f  (should be ~0)' % null['t_nw'])
print('planted world: spread NW t = %+.2f  (should light up above |t|=2)' % planted['t_nw'])

null world   : spread NW t = -0.98  (should be ~0)
planted world: spread NW t = +2.52  (should light up above |t|=2)


## 3. The honest verdict — the premium does *not* carry to asset classes

On the nine-class tape the long-low-skew / short-high-skew spread is **+13.73 bps/month** with NW *t* = **+0.62** — the *sign* is in the claimed direction (low-skew classes did edge out high-skew ones), but the magnitude is **statistically zero**. A 1,000-permutation placebo puts it only ~0.8σ into the right tail (p = 0.20), and the spread even *flips sign* across the two eras (-15 bps early, +40 bps late). With just nine classes there is too little skew dispersion for the effect to exist. The seeded synthetic control shows a real premium of plausible size *would* have fired, so this is an honest null, not a broken sort. **Signal: None**, **Tradability: Mirage**.